In [120]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
from typing import Any, Dict, List
from sqlalchemy import Integer, JSON, String, Text, func, text
from sqlalchemy.orm import DeclarativeBase, declared_attr, Mapped, mapped_column
from sqlalchemy.ext.asyncio import AsyncAttrs, async_sessionmaker, create_async_engine
from pgvector.sqlalchemy import Vector
from langchain_openai import ChatOpenAI
from langchain_gigachat.chat_models import GigaChat
from langchain_gigachat import GigaChatEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

import asyncio
import numpy as np
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langgraph.prebuilt import ToolNode, tools_condition

import uuid
from datetime import datetime
from typing import Any, Dict, List, Optional
from sqlalchemy import JSON, Column, Integer, String, Boolean, DateTime, ForeignKey, Text
from sqlalchemy.orm import Mapped, mapped_column, relationship
from sqlalchemy.sql import func
from sqlalchemy.dialects.postgresql import ARRAY, UUID

from typing import List, Optional, Union, Annotated
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langgraph.graph.message import add_messages

In [121]:
load_dotenv()

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")

POSTGRES_DB_NAME = "userdb"

POSTGRES_URL = (
    f"postgresql+asyncpg://user"
    f":user"
    f"@localhost"
    f":5436"
    f"/{POSTGRES_DB_NAME}"
)

In [122]:
from langchain_openai import ChatOpenAI
from langchain_gigachat import GigaChat

# main_llm = GigaChat(
#     model="GigaChat:latest",
#     credentials=GIGACHAT_API_KEY,
#     scope = "GIGACHAT_API_B2B",
#     verify_ssl_certs=False,
# )

main_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model="gpt-4o-mini",
    temperature=0.7
)

In [ ]:
async def assistant(state):
    response = await main_llm.ainvoke([SystemMessage(content=system_prompt)] + state.messages)

    answer_arm_data = response.response_metadata['token_usage']
    answer_aum_data = response.usage_metadata

    prompt_tokens = answer_arm_data.get('prompt_tokens') or 0
    total_tokens = answer_arm_data.get('total_tokens') or 0
    cache_tokens = answer_arm_data.get('precached_prompt_tokens') or 0

    return {"messages": [response]}

In [124]:
class TaskState(BaseModel):
    # messages: Annotated[list, add_messages]
    query: str
    topic: str
    level: int
    dout: Optional[List[str]] = None
    MAX_LEVEL: int = 10

    task_content: Optional[str] = None
    task_answer: Optional[str] = None


class UserBlock(BaseModel):
    content: str = Field(default="# ВАШЕ РЕШЕНИЕ ЗДЕСЬ")

class ExpectedOutput(BaseModel):
    content: str

class Task(BaseModel):
    id: int
    title: str
    description: str
    difficulty: str
    user_block: UserBlock
    expected_output: ExpectedOutput

class TaskDocument(BaseModel):
    """
    Документ с набором учебных заданий для системы проверки знаний.
    
    Содержит метаданные набора заданий и список конкретных задач.
    Используется для генерации интерактивных MD файлов с полями для ответов пользователя.
    """
    title: str = Field( max_length=100, description="""Название набора заданий.  Должно быть кратким и информативным (например:  'Практика Python asyncio', 'Английский B1 - Грамматика').""")
    topic: str = Field(max_length=50, description="""Тема/предмет заданий. Краткое название дисциплины или раздела (например: 'Python asyncio', 'Английский язык', 'Математика').""")
    level: int = Field( ge=1, le=10, description="""Уровень сложности текущего набора заданий. Целое число от 1 (базовый) до 10 (экспертный). Используется для адаптивной генерации контента.""")
    max_level: int = Field( default=10, ge=1, le=20, description="""Максимальный уровень сложности в системе. Используется для отображения прогресса (например: 'уровень 5/10'). Обычно равен 10, но может быть настроен.""")
    tasks: List[Task] = Field(min_items=1, max_items=10, description="""Список заданий в наборе. Каждое задание содержит: - Уникальный ID - Название и описание - Уровень сложности - Пустое поле для ответа (user_block) - Ожидаемый результат (expected_output)""")

C:\Users\Tumbi\AppData\Local\Temp\ipykernel_22136\4212976811.py:38: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  tasks: List[Task] = Field(min_items=1, max_items=10, description="""Список заданий в наборе. Каждое задание содержит: - Уникальный ID - Название и описание - Уровень сложности - Пустое поле для ответа (user_block) - Ожидаемый результат (expected_output)""")
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_22136\4212976811.py:38: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  tasks: List[Task] = Field(min_items=1, max_items=10, description="""Список заданий в наборе. Каждое задание содержит: - Уникальный ID - Назван

In [125]:
from pydantic import BaseModel, Field
from typing import List
from jinja2 import Template
import json

TASK_TEMPLATE = Template("""
# {{ title }}
**Тема:** {{ topic }} | **Уровень:** {{ level }}/{{ max_level }}

{% for task in tasks %}
## Задание {{ task.id }}: {{ task.title }}

**Сложность:** {{ task.difficulty.title() }}

**Условие:**
{{ task.description }}

**Ваше решение:**
```
{{ task.user_block.content }}
```
                        
{% endfor %}
                        
<i>Отправьте заполненный файл на проверку командой /check</i>
""")

In [126]:
data_dir_path = "./data"
task_dir_path = os.path.join(data_dir_path, "tasks")
os.makedirs(task_dir_path, exist_ok=True)


async def generate_task(state: TaskState):
    try:
        prompt = ChatPromptTemplate.from_template(
            ("Ты эксперт в области {topic}."
            "Составь {level_count} заданий с уровнем навыка {level}/{max_level}."
            "Верни СТРОГО JSON по схеме:\n{schema}"
            "НЕ ЗАПОЛНЯТЬ БЛОК 'Ваше решение'")).partial(schema=TaskDocument.model_json_schema())

        chain = prompt | main_llm | JsonOutputParser(pydantic_object=TaskDocument)
        task_content = await chain.ainvoke({
            "topic": state.topic,
            "level": state.level,
            "level_count": 3,
            "max_level": state.MAX_LEVEL,
            "query": state.query
        })
        # data = json.loads(task_content)
        task_doc = TaskDocument.model_validate(task_content)
        task_md_content = TASK_TEMPLATE.render(task_doc.model_dump())

        from IPython.display import Markdown, display
        display(Markdown(task_md_content))

        filename = f"task_{state.topic}_{state.level}.md"
        filepath = Path(task_dir_path) / filename
        filepath.write_text(task_md_content, encoding="utf-8")
        return {'dout': [f"Сгенерировано: {filename}", f"📁 Путь: {filepath}"], 'task_content': task_md_content}
    except Exception as e:
        return {'dout': f"error: {e}", 'task_content': task_content}


async def resume_task():
    try:
        template_file = Path(task_dir_path) / "task_Английский язык_3.md" # filename
        content = template_file.read_text(encoding="utf-8")
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Ты эксперт-преподаватель. Проверь решение студента:
            
            Критерии оценки (0-10 баллов):
            - Корректность логики (4 балла)
            - Полное решение задачи (3 балла)  
            - Чистый код и стиль (2 балла)
            - Эффективность (1 балл)
            
            Верни ТОЛЬКО оценку в формате:
            ## Задание
            Оценка: X/10
            ✅ Верно / ❌ Ошибки: [кратко]
            Комментарий: [полезная обратная связь]
            """),
            ("human", content)
        ])
        chain = prompt | main_llm
        result = await chain.ainvoke({})
        return {'dout': f"Ваша оценка:\n{result}"}
    except Exception as e:
        return {'dout': f"error: {e}"}


graph = StateGraph(TaskState)

graph.add_node("gen_task", generate_task)

graph.add_edge(START, "gen_task")
graph.add_edge("gen_task", END)

app = graph.compile()

In [127]:
class TaskData(BaseModel):
    id: int
    content: str
    answer: Optional[str] = None


tasks: List[TaskData] = []

In [128]:
topic = "Английский язык"
level = 3

res = await app.ainvoke({
    "query": f"Задание по {topic} с уровнем навыка {level}",
    "topic": topic,
    "level": level
})

# tasks.append(TaskData(id=1, content=res['task_content']))
print(res['dout'])


# Английский язык - Уровень A1
**Тема:** Английский язык | **Уровень:** 3/10


## Задание 1: Заполните пропуски

**Сложность:** Легкий

**Условие:**
Дополните предложения, используя правильные формы глаголов 'to be'.

**Ваше решение:**
```
# ВАШЕ РЕШЕНИЕ ЗДЕСЬ
```


## Задание 2: Переведите слова

**Сложность:** Легкий

**Условие:**
Переведите следующие слова на английский: 'кот', 'собака', 'машина'.

**Ваше решение:**
```
# ВАШЕ РЕШЕНИЕ ЗДЕСЬ
```


## Задание 3: Составьте предложения

**Сложность:** Легкий

**Условие:**
Используйте слова 'I', 'like', 'apples' для составления предложения.

**Ваше решение:**
```
# ВАШЕ РЕШЕНИЕ ЗДЕСЬ
```



<i>Отправьте заполненный файл на проверку командой /check</i>

['Сгенерировано: task_Английский язык_3.md', '📁 Путь: data\\tasks\\task_Английский язык_3.md']


In [129]:
res = await resume_task()
res['dout']

"Ваша оценка:\ncontent='## Задание\\nОценка: 0/10\\n❌ Ошибки: Решение отсутствует\\nКомментарий: Пожалуйста, заполните все задания, предоставив свои ответы, чтобы я мог оценить вашу работу.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 409, 'total_tokens': 460, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 9.195e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 9.195e-05, 'upstream_inference_prompt_cost': 6.135e-05, 'upstream_inference_completions_cost': 3.06e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'gen-1773140987-Dm1t0U7gEcG7rB8bziAa', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cd